#Setup

In [ ]:
%cd /content/drive/MyDrive/263

/content/drive/MyDrive/263


In [1]:
from google.colab import userdata
hf_token = userdata.get('hugging_face')
openai_token = userdata.get('openai')

ModuleNotFoundError: No module named 'google'

In [ ]:
hf_token = ""
openai_token = ""

In [ ]:
#import dataset
import pandas as pd

df = pd.read_csv('/Users/Suhas/Social-Acceptability-Classification/Dataset/CS263_dataset.csv')
df.head()

,id,category,interaction,culture,label,confidence_score,cultural_context
0,SAC001,family,User: I didn’t tell my parents I moved to anot...,"US, adult independence norm",acceptable,0.86,Adult autonomy and privacy are commonly respec...
1,SAC002,family,User: I didn’t tell my parents I moved to anot...,"China, family-involvement norm",context-dependent,0.78,Major life decisions are often shared with fam...
2,SAC003,family,User: I skipped a family dinner because I want...,"US, personal-boundary norm",acceptable,0.84,Personal time and emotional boundaries are gen...
3,SAC004,family,User: I skipped a family dinner because I want...,"Italy, family-gathering norm",context-dependent,0.76,"Family gatherings can be socially important, b..."
4,SAC005,family,User: I didn’t help my sibling financially.\nL...,"US, financial-independence norm",context-dependent,0.80,Individual financial responsibility is emphasi...


#DeBERTa 2 Class + Threshold
We chose DeBERTa as one of our base models because its a model that is optimized for Natural Language Inference, trained to handle entailment-style reasoning.

**Design**

Have DeBERTa predict between "acceptable" and "not acceptable", if the confidence is below a threshold, return "context-dependent" instead.

In [3]:
import re
import torch
import pandas as pd
from transformers import pipeline

# 1. Load dataset
df = pd.read_csv('/Users/Suhas/Social-Acceptability-Classification/Dataset/CS263_dataset.csv')

# 2. Parse User sentence
def parse_user_interaction(text):
    match = re.search(r"User:\s*(.*?)(?:\\n|\n|$)", text)
    return match.group(1).strip() if match else None

df["situation"] = df["interaction"].apply(parse_user_interaction)

# 3. Load DeBERTa MNLI model
classifier = pipeline(
    "zero-shot-classification",
    model="microsoft/deberta-large-mnli",
    token=hf_token
)

candidate_labels = ["acceptable behavior", "unacceptable behavior"]

def predict_acceptability(situation, threshold=0.65):
    result = classifier(
        situation,
        candidate_labels=[
            "acceptable behavior",
            "not acceptable behavior"
        ],
        hypothesis_template="This behavior is {}."
    )

    top_label = result["labels"][0]
    top_score = result["scores"][0]

    if top_score < threshold:
        return "context-dependent"

    if top_label == "acceptable behavior":
        return "acceptable"
    else:
        return "not acceptable"

# 5. Predict
predictions = []

for _, row in df.iterrows():

    with_context = f"{row['cultural_context']} {row['situation']}"
    pred_label = predict_acceptability(with_context)

    # pred_label = predict_acceptability(row["situation"])

    predictions.append({
        "id": row["id"],
        "situation": row["situation"],
        "gold_label": row["label"],
        "prediction_label": pred_label
    })

# 6. Create result DataFrame
results_df = pd.DataFrame(predictions)

results_df.head()
results_df.to_csv("deberta_predictions_with_context.csv", index=False)

/Users/Suhas/Social-Acceptability-Classification/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 392/392 [00:00<00:00, 70209.55it/s]
[transformers] DebertaForSequenceClassification LOAD REPORT from: microsoft/deberta-large-mnli
Key    | Status     |  | 
-------+------------+--+-
config | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] Error during conversion: ReadTimeout('The read operation timed out')
Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.12/3.12.7_1/Frameworks/Python.framework/Versions/3.12/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/opt/hom

In [5]:
!pip install openai

  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.46.4-cp312-cp312-macosx_11_0_arm64.whl.metadata (6.6 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 6.2 MB/s  0:00:00
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
Using cached pydantic-2.13.4-py3-none-any.whl (472 kB)
Using cached pydantic_core-2.46.4-cp312-cp312-macosx_11_0_arm64.whl (2.0 MB)
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [openai]2m7/8 [openai]c]


#ChatGPT

In [4]:
from openai import OpenAI
import json
import pandas as pd

client = OpenAI(api_key=openai_token)

def predict_acceptability_gpt(situation, cultural_context, culture):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": """
You are a social acceptability classifier.

Classify the user's situation into exactly one label:
- acceptable
- not acceptable
- context-dependent

Use the provided cultural context to inform your classification.
Return only JSON in this format:
{"label": "..."}
"""
            },
            {
                "role": "user",
                "content": f"Situation: {situation}\nCultural context: {cultural_context}\nCulture: {culture}"
            }
        ]
    )

    text = response.choices[0].message.content
    return json.loads(text)["label"]

In [7]:
gpt_predictions = []

for _, row in df.iterrows():
    pred_label = predict_acceptability_gpt(row["situation"], row["cultural_context"], row["culture"])
    # pred_label = predict_acceptability_gpt(row["situation"])

    gpt_predictions.append({
        "id": row["id"],
        "situation": row["situation"],
        "gold_label": row["label"],
        "prediction_label": pred_label
    })

gpt_results_df = pd.DataFrame(gpt_predictions)

gpt_results_df.head()
gpt_results_df.to_csv("gpt_predictions_with_context.csv", index=False)

#Accuracy Analysis

In [8]:
from sklearn.metrics import accuracy_score

# DeBERTa
deberta_acc = accuracy_score(
    results_df["gold_label"],
    results_df["prediction_label"]
)

# GPT
gpt_acc = accuracy_score(
    gpt_results_df["gold_label"],
    gpt_results_df["prediction_label"]
)

print(f"DeBERTa Accuracy: {deberta_acc:.4f}")
print(f"GPT Accuracy: {gpt_acc:.4f}")

DeBERTa Accuracy: 0.6250
GPT Accuracy: 0.7917


In [9]:
with open("accuracy_summary_with_context.txt", "w") as f:
    f.write(f"DeBERTa Accuracy: {deberta_acc:.4f}\n")
    f.write(f"GPT Accuracy: {gpt_acc:.4f}\n")

#Evaluation

In [10]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

labels = ["acceptable", "unacceptable", "context-dependent"]

def evaluate_model(df, model_name, save_path):
    y_true = df["gold_label"]
    y_pred = df["prediction_label"]

    # Accuracy
    acc = accuracy_score(y_true, y_pred)

    # Classification report
    report = classification_report(y_true, y_pred, labels=labels)

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    # Print results
    print(f"\n===== {model_name} =====")
    print(f"Accuracy: {acc:.4f}\n")
    print("Classification Report:")
    print(report)
    print("Confusion Matrix:")
    print(cm)

    # Save to file
    with open(save_path, "w") as f:
        f.write(f"===== {model_name} =====\n")
        f.write(f"Accuracy: {acc:.4f}\n\n")
        f.write("Classification Report:\n")
        f.write(report + "\n")
        f.write("Confusion Matrix:\n")
        f.write(str(cm))

# Run evaluations
evaluate_model(results_df, "DeBERTa", "deberta_eval_with_context.txt")
evaluate_model(gpt_results_df, "GPT", "gpt_eval_with_context.txt")


===== DeBERTa =====
Accuracy: 0.6250

Classification Report:
                   precision    recall  f1-score   support

       acceptable       0.42      0.87      0.56        23
     unacceptable       0.00      0.00      0.00         0
context-dependent       0.79      0.32      0.45        47

        micro avg       0.52      0.50      0.51        70
        macro avg       0.40      0.40      0.34        70
     weighted avg       0.67      0.50      0.49        70

Confusion Matrix:
[[20  0  0]
 [ 0  0  0]
 [22  0 15]]

===== GPT =====
Accuracy: 0.7917

Classification Report:
                   precision    recall  f1-score   support

       acceptable       0.71      0.96      0.81        23
     unacceptable       0.00      0.00      0.00         0
context-dependent       1.00      0.49      0.66        47

        micro avg       0.83      0.64      0.73        70
        macro avg       0.57      0.48      0.49        70
     weighted avg       0.90      0.64      0.71     

/Users/Suhas/Social-Acceptability-Classification/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/Suhas/Social-Acceptability-Classification/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/Suhas/Social-Acceptability-Classification/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to contro

Aggregate Result

In [11]:
import pandas as pd

df = pd.read_csv("/Users/Suhas/Social-Acceptability-Classification/Dataset/CS263_dataset.csv")

deberta_df = pd.read_csv("/Users/Suhas/Social-Acceptability-Classification/deberta_predictions_with_context.csv")
gpt_df = pd.read_csv("/Users/Suhas/Social-Acceptability-Classification/gpt_predictions_with_context.csv")

deberta_df = deberta_df.rename(columns={"prediction_label": "deberta_label"})
gpt_df = gpt_df.rename(columns={"prediction_label": "gpt_label"})

df = df.merge(
    deberta_df[["id", "situation", "deberta_label"]],
    on="id",
    how="left"
)

df = df.merge(
    gpt_df[["id", "gpt_label"]],
    on="id",
    how="left"
)

df.to_csv("/Users/Suhas/Social-Acceptability-Classification/Dataset/CS263_dataset_with_predictions.csv", index=False)

In [12]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

# ===== 1. Load CSV =====
input_file = "/Users/Suhas/Social-Acceptability-Classification/Dataset/CS263_dataset_with_predictions.csv"
df = pd.read_csv(input_file)

# ===== 2. Extract country from culture column =====
# Example: "US, adult independence norm" -> "US"
df["country"] = df["culture"].str.split(",").str[0].str.strip()

# ===== 3. Normalize labels =====
label_cols = ["label", "deberta_label", "gpt_label"]

for col in label_cols:
    df[col] = df[col].astype(str).str.lower().str.strip()

# ===== 4. Accuracy grouped by country =====
summary_rows = []

for country, group in df.groupby("country"):
    deberta_acc = accuracy_score(group["label"], group["deberta_label"])
    gpt_acc = accuracy_score(group["label"], group["gpt_label"])

    summary_rows.append({
        "country": country,
        "n_samples": len(group),
        "deberta_accuracy": deberta_acc,
        "gpt_accuracy": gpt_acc
    })

summary_df = pd.DataFrame(summary_rows).sort_values("country")

print("\n=== Accuracy by Country ===")
print(summary_df)

# ===== 5. Overall accuracy =====
overall = pd.DataFrame([
    {
        "model": "DeBERTa",
        "accuracy": accuracy_score(df["label"], df["deberta_label"])
    },
    {
        "model": "GPT",
        "accuracy": accuracy_score(df["label"], df["gpt_label"])
    }
])

print("\n=== Overall Accuracy ===")
print(overall)

# ===== 6. Full classification report by country =====
for country, group in df.groupby("country"):
    print(f"\n\n================ {country} ================")

    print("\n--- DeBERTa Classification Report ---")
    print(classification_report(
        group["label"],
        group["deberta_label"],
        zero_division=0
    ))

    print("\n--- GPT Classification Report ---")
    print(classification_report(
        group["label"],
        group["gpt_label"],
        zero_division=0
    ))

# ===== 7. Save country-level summary =====
summary_df.to_csv("accuracy_by_country_with_context.csv", index=False)
overall.to_csv("overall_accuracy_with_context.csv", index=False)

print("\nSaved:")
print("- accuracy_by_country_with_context.csv")
print("- overall_accuracy_with_context.csv")


=== Accuracy by Country ===
        country  n_samples  deberta_accuracy  gpt_accuracy
0        Brazil          4          0.250000      0.500000
1         China         13          0.538462      0.846154
2       Finland          1          1.000000      1.000000
3        France          2          1.000000      1.000000
4       Germany         12          0.666667      0.833333
5         India          5          0.400000      0.600000
6         Italy          2          0.500000      1.000000
7         Japan         17          0.764706      0.823529
8         Korea          3          1.000000      1.000000
9        Mexico          1          1.000000      0.000000
10  Netherlands          1          1.000000      1.000000
11  South Korea          2          0.500000      1.000000
12  Southern US          1          0.000000      0.000000
13     Thailand          3          1.000000      1.000000
14           UK          1          0.000000      1.000000
15           US         51 